# Project 05 — Customer Segmentation & RFM Analysis
This notebook validates transaction data, calculates RFM metrics, creates customer segments, and identifies retention priorities.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('../data/customer_transactions.csv', parse_dates=['Order_Date'])
df.head()

## 1. Data validation

In [ ]:
print('Shape:', df.shape)
print('Missing values:', int(df.isna().sum().sum()))
print('Duplicate order IDs:', int(df['Order_ID'].duplicated().sum()))
print(df[['Revenue_MAD','Cost_MAD','Profit_MAD']].describe())

## 2. Customer-level RFM metrics

In [ ]:
analysis_date = df['Order_Date'].max() + pd.Timedelta(days=1)
rfm = df.groupby('Customer_ID').agg(Last_Order=('Order_Date','max'), Frequency=('Order_ID','nunique'), Monetary=('Revenue_MAD','sum')).reset_index()
rfm['Recency'] = (analysis_date - rfm['Last_Order']).dt.days
rfm.head()

## 3. RFM scoring

In [ ]:
rfm['R_Score'] = pd.qcut(rfm['Recency'].rank(method='first', ascending=True), 5, labels=[5,4,3,2,1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['RFM_Score'] = rfm[['R_Score','F_Score','M_Score']].sum(axis=1)

## 4. Business segmentation

In [ ]:
def segment(r):
    if r.R_Score >= 4 and r.F_Score >= 4 and r.M_Score >= 4: return 'Champions'
    if r.R_Score >= 4 and r.F_Score >= 3: return 'Loyal Customers'
    if r.R_Score <= 2 and r.F_Score >= 3 and r.M_Score >= 3: return 'At Risk - High Value'
    if r.R_Score <= 2: return 'At Risk'
    return 'Potential Loyalists'

rfm['Segment'] = rfm.apply(segment, axis=1)
segment_summary = rfm.groupby('Segment').agg(Customers=('Customer_ID','count'), Revenue=('Monetary','sum'), Avg_Recency=('Recency','mean'), Avg_Frequency=('Frequency','mean')).sort_values('Revenue', ascending=False)
segment_summary

## 5. Revenue by segment

In [ ]:
segment_summary['Revenue'].plot(kind='bar', title='Revenue by Customer Segment')
plt.ylabel('Revenue (MAD)')
plt.tight_layout()
plt.show()

## 6. Retention targets

In [ ]:
retention_targets = rfm[rfm['Segment'].eq('At Risk - High Value')].sort_values('Monetary', ascending=False)
retention_targets[['Customer_ID','Recency','Frequency','Monetary','Segment']].head(20)

## Conclusion
Champions should be protected, At Risk - High Value customers prioritized for reactivation, and Potential Loyalists developed toward higher frequency. These are descriptive priorities, not causal predictions.